In [ ]:
# Cell 1 — Installs and imports
!pip -q install -U torch transformers datasets accelerate matplotlib tqdm

import copy
import json
import math
import os
import random
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)


In [ ]:
# Cell 2 — Config
@dataclass
class UTAConfig:
    model_name: str = "EleutherAI/pythia-410m"
    output_dir: str = "./artifacts"
    checkpoint_name: str = "uta_pythia410m.pt"
    random_seed: int = 42

    rank: int = 16
    batch_size: int = 2
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    fine_tune_epochs: int = 1
    warmup_ratio: float = 0.06
    log_every_n_steps: int = 20

    max_seq_len: int = 256
    train_subset_size: int = 2000
    eval_subset_per_hop: int = 150

    musique_max_new_tokens: int = 24
    generation_temperature: float = 0.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = UTAConfig()
config = asdict(cfg)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

random.seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
torch.manual_seed(cfg.random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.random_seed)

print(json.dumps(config, indent=2))


In [ ]:
# Cell 3 — Load Pythia-410m

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(cfg.device)
base_model.eval()

for i, layer in enumerate(base_model.gpt_neox.layers):
    print(f"Layer {i}: {layer.attention.__class__.__name__}")

for p in base_model.parameters():
    p.requires_grad = False

print("Total params:", sum(p.numel() for p in base_model.parameters()))
print("Trainable params:", sum(p.numel() for p in base_model.parameters() if p.requires_grad))


In [ ]:
# Cell 4 — UTA attention module

def factorize_weight_svd(weight: torch.Tensor, rank: int) -> Tuple[torch.Tensor, torch.Tensor]:
    out_features, in_features = weight.shape
    used_rank = min(rank, out_features, in_features)
    u, s, v_h = torch.linalg.svd(weight, full_matrices=False)
    u_r = u[:, :used_rank]
    s_r = s[:used_rank]
    v_r = v_h[:used_rank, :]
    left = (u_r * s_r).contiguous()
    right = v_r.contiguous()
    if used_rank < rank:
        left_pad = torch.zeros(out_features, rank - used_rank, device=weight.device, dtype=weight.dtype)
        right_pad = torch.zeros(rank - used_rank, in_features, device=weight.device, dtype=weight.dtype)
        left = torch.cat([left, left_pad], dim=1)
        right = torch.cat([right, right_pad], dim=0)
    return right.t().contiguous(), left.contiguous()


class LowRankLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.A = nn.Parameter(torch.empty(in_features, rank))
        self.B = nn.Parameter(torch.empty(rank, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.A)
        nn.init.xavier_uniform_(self.B)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = x @ self.A
        y = y @ self.B
        if self.bias is not None:
            y = y + self.bias
        return y

    @torch.no_grad()
    def load_from_dense(self, dense_weight: torch.Tensor, dense_bias: Optional[torch.Tensor] = None):
        A, B = factorize_weight_svd(dense_weight, self.rank)
        self.A.copy_(A)
        self.B.copy_(B)
        if self.bias is not None and dense_bias is not None:
            self.bias.copy_(dense_bias)


class UTAAttention(nn.Module):
    def __init__(self, original_attention: nn.Module, rank: int):
        super().__init__()
        self.hidden_size = original_attention.hidden_size
        self.num_attention_heads = original_attention.num_attention_heads
        self.head_size = self.hidden_size // self.num_attention_heads
        self.rank = rank

        self.query = LowRankLinear(self.hidden_size, self.hidden_size, rank, bias=True)
        self.key = LowRankLinear(self.hidden_size, self.hidden_size, rank, bias=True)
        self.value = LowRankLinear(self.hidden_size, self.hidden_size, rank, bias=True)
        self.dense = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.u_mean = nn.Linear(self.hidden_size, self.hidden_size)
        self.u_var = nn.Linear(self.hidden_size, self.hidden_size)
        self.uncertainty_axis_attn = nn.Parameter(torch.eye(3))

        self.norm_factor = math.sqrt(self.head_size)
        self.register_buffer("latest_uncertainty", torch.zeros(1), persistent=False)

        self._copy_from_original(original_attention)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        b, s, _ = x.shape
        return x.view(b, s, self.num_attention_heads, self.head_size).permute(0, 2, 1, 3)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        b, h, s, d = x.shape
        return x.permute(0, 2, 1, 3).contiguous().view(b, s, h * d)

    @torch.no_grad()
    def _copy_from_original(self, original_attention: nn.Module):
        qkv_w = original_attention.query_key_value.weight.data
        qkv_b = original_attention.query_key_value.bias.data
        q_w, k_w, v_w = torch.chunk(qkv_w, 3, dim=0)
        q_b, k_b, v_b = torch.chunk(qkv_b, 3, dim=0)

        self.query.load_from_dense(q_w, q_b)
        self.key.load_from_dense(k_w, k_b)
        self.value.load_from_dense(v_w, v_b)

        self.dense.weight.copy_(original_attention.dense.weight.data)
        self.dense.bias.copy_(original_attention.dense.bias.data)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
        output_attentions: bool = False,
        padding_mask: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        b, s, _ = hidden_states.shape

        q = self._split_heads(self.query(hidden_states))
        k = self._split_heads(self.key(hidden_states))
        v = self._split_heads(self.value(hidden_states))

        if layer_past is not None:
            past_k, past_v = layer_past
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        present = (k, v) if use_cache else None

        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / self.norm_factor

        causal_mask = torch.tril(torch.ones((s, k.size(2)), device=hidden_states.device, dtype=torch.bool))
        causal_mask = causal_mask.view(1, 1, s, k.size(2))
        attn_scores = attn_scores.masked_fill(~causal_mask, torch.finfo(attn_scores.dtype).min)

        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask

        attn_probs = F.softmax(attn_scores, dim=-1)
        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        main_ctx = torch.matmul(attn_probs, v)
        main_ctx_merged = self._merge_heads(main_ctx)

        u_mean = self.u_mean(hidden_states)
        u_var = F.softplus(self.u_var(hidden_states)) + 1e-6

        main_heads = main_ctx_merged.view(b, s, self.num_attention_heads, self.head_size)
        mean_heads = u_mean.view(b, s, self.num_attention_heads, self.head_size)
        var_heads = u_var.view(b, s, self.num_attention_heads, self.head_size)

        stream_tensor = torch.stack([main_heads, mean_heads, var_heads], dim=2)
        axis_weights = F.softmax(self.uncertainty_axis_attn, dim=-1)
        mixed = torch.einsum('ij,bsjhd->bsihd', axis_weights, stream_tensor)

        main_mixed = mixed[:, :, 0].contiguous().view(b, s, self.hidden_size)
        var_mixed = mixed[:, :, 2].contiguous().view(b, s, self.hidden_size)

        uncertainty_scalar = var_mixed.mean(dim=-1)
        self.latest_uncertainty = uncertainty_scalar.detach()

        attn_output = self.dense(main_mixed)

        if output_attentions:
            return attn_output, present, attn_probs
        return attn_output, present


In [ ]:
# Cell 5 — Graft UTA into Pythia-410m
uta_model = copy.deepcopy(base_model).to(cfg.device)
uta_model.eval()

for p in uta_model.parameters():
    p.requires_grad = False

for i, layer in enumerate(uta_model.gpt_neox.layers):
    original_attention = layer.attention
    layer.attention = UTAAttention(original_attention, rank=cfg.rank).to(cfg.device)
    for p in layer.attention.parameters():
        p.requires_grad = True
    print(f"Replaced layer {i} attention with UTAAttention")

frozen_params = sum(p.numel() for p in uta_model.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in uta_model.parameters() if p.requires_grad)
print(f"Frozen params: {frozen_params:,}")
print(f"Trainable params: {trainable_params:,}")


In [ ]:
# Cell 6 — Fine-tuning loop

def tokenize_lm_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=cfg.max_seq_len,
        padding="max_length",
    )

wikitext = load_dataset("wikitext", "wikitext-2-raw-v1")
train_ds = wikitext["train"].filter(lambda x: len(x["text"].strip()) > 0)
if cfg.train_subset_size > 0:
    train_ds = train_ds.select(range(min(cfg.train_subset_size, len(train_ds))))

train_tok = train_ds.map(tokenize_lm_batch, batched=True, remove_columns=train_ds.column_names)
train_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])

train_loader = DataLoader(train_tok, batch_size=cfg.batch_size, shuffle=True)

optimizer = AdamW([p for p in uta_model.parameters() if p.requires_grad], lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
total_steps = max(1, cfg.fine_tune_epochs * len(train_loader))
warmup_steps = int(total_steps * cfg.warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

uta_model.train()
global_step = 0
for epoch in range(cfg.fine_tune_epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.fine_tune_epochs}")
    for batch in pbar:
        batch = {k: v.to(cfg.device) for k, v in batch.items()}
        labels = batch["input_ids"].clone()

        out = uta_model(**batch, labels=labels)
        loss = out.loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()

        global_step += 1
        if global_step % cfg.log_every_n_steps == 0:
            pbar.set_postfix(loss=float(loss.detach().cpu()))

ckpt_path = Path(cfg.output_dir) / cfg.checkpoint_name
torch.save({"state_dict": uta_model.state_dict(), "config": asdict(cfg)}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")


In [ ]:
# Cell 7 — Load MuSiQue
musique = load_dataset("alexwei/musique", split="validation")

def infer_hop(example):
    if "num_hops" in example and example["num_hops"] is not None:
        return int(example["num_hops"])
    if "decomposition" in example and example["decomposition"] is not None:
        return len(example["decomposition"])
    if "question_decomposition" in example and example["question_decomposition"] is not None:
        return len(example["question_decomposition"])
    return None

hop_buckets: Dict[int, List[Dict]] = {2: [], 3: [], 4: []}
for row in musique:
    hop = infer_hop(row)
    if hop in hop_buckets:
        hop_buckets[hop].append(row)

for hop in hop_buckets:
    hop_buckets[hop] = hop_buckets[hop][: cfg.eval_subset_per_hop]

print({f"{k}-hop": len(v) for k, v in hop_buckets.items()})

def format_prompt(sample: Dict) -> str:
    contexts = sample.get("paragraphs", [])
    if isinstance(contexts, list):
        context_text = "\n".join([c.get("paragraph_text", str(c)) if isinstance(c, dict) else str(c) for c in contexts])
    else:
        context_text = str(contexts)
    q = sample.get("question", "")
    return f"Context:\n{context_text}\n\nQuestion: {q}\nAnswer:"

hop_loaders = {}
for hop, samples in hop_buckets.items():
    prompts = [format_prompt(s) for s in samples]
    answers = [s.get("answer", "") for s in samples]
    hop_loaders[hop] = list(zip(prompts, answers))


In [ ]:
# Cell 8 — Evaluation loop

def normalize_text(text: str) -> str:
    return " ".join(text.lower().strip().split())

@torch.no_grad()
def generate_answer(model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=cfg.max_seq_len).to(cfg.device)
    generated = model.generate(
        **inputs,
        max_new_tokens=cfg.musique_max_new_tokens,
        do_sample=False if cfg.generation_temperature == 0 else True,
        temperature=max(cfg.generation_temperature, 1e-6),
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_text = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen_text.strip()

@torch.no_grad()
def get_uta_uncertainty(model, prompt: str) -> float:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=cfg.max_seq_len).to(cfg.device)
    _ = model(**inputs)
    final_unc = model.gpt_neox.layers[-1].attention.latest_uncertainty
    return float(final_unc.mean().item())

base_model.eval()
uta_model.eval()

results = {}
for hop, items in hop_loaders.items():
    base_correct = []
    uta_correct = []
    uta_unc = []

    for prompt, gold in tqdm(items, desc=f"Eval {hop}-hop"):
        gold_norm = normalize_text(gold)

        base_pred = normalize_text(generate_answer(base_model, prompt))
        uta_pred = normalize_text(generate_answer(uta_model, prompt))
        unc_val = get_uta_uncertainty(uta_model, prompt)

        base_ok = int(base_pred == gold_norm)
        uta_ok = int(uta_pred == gold_norm)

        base_correct.append(base_ok)
        uta_correct.append(uta_ok)
        uta_unc.append(unc_val)

    results[hop] = {
        "base_em": float(np.mean(base_correct)) if base_correct else 0.0,
        "uta_em": float(np.mean(uta_correct)) if uta_correct else 0.0,
        "uta_uncertainty_mean": float(np.mean(uta_unc)) if uta_unc else 0.0,
        "per_sample": {
            "base_correct": base_correct,
            "uta_correct": uta_correct,
            "uta_uncertainty": uta_unc,
        },
    }

print(json.dumps(results, indent=2))


In [ ]:
# Cell 9 — Plots
hop_counts = sorted(results.keys())
base_acc = [results[h]["base_em"] for h in hop_counts]
uta_acc = [results[h]["uta_em"] for h in hop_counts]
uta_unc = [results[h]["uta_uncertainty_mean"] for h in hop_counts]

plt.figure(figsize=(8, 5))
plt.plot(hop_counts, base_acc, marker="o", label="Base Pythia-410m")
plt.plot(hop_counts, uta_acc, marker="o", label="UTA-grafted Pythia-410m")
plt.xlabel("Hop count")
plt.ylabel("Exact Match Accuracy")
plt.title("Accuracy vs Hop Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("accuracy_vs_hop.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(hop_counts, uta_unc, marker="o", color="tab:orange")
plt.xlabel("Hop count")
plt.ylabel("Mean UTA uncertainty signal")
plt.title("UTA Uncertainty vs Hop Count")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("uncertainty_vs_hop.png", dpi=150)
plt.show()

all_unc = []
all_correct = []
for h in hop_counts:
    all_unc.extend(results[h]["per_sample"]["uta_uncertainty"])
    all_correct.extend(results[h]["per_sample"]["uta_correct"])

plt.figure(figsize=(8, 5))
plt.scatter(all_unc, all_correct, alpha=0.55)
plt.xlabel("Per-sample UTA uncertainty")
plt.ylabel("Correctness (1 = exact match)")
plt.title("Uncertainty vs Correctness")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("uncertainty_vs_correctness_scatter.png", dpi=150)
plt.show()


In [ ]:
# Cell 10 — GitHub PR
# Fill this URL before running on your machine.
REPO_URL = "https://github.com/<your-org>/<your-repo>.git"
BRANCH_NAME = "uta-attention-v1"
PR_TITLE = "UTA Attention v1 — 3rd order tensor uncertainty attention grafted onto Pythia-410m"
PR_BODY = (
    "Implemented a fully executable UTA v1 notebook that loads pretrained Pythia-410m, "
    "replaces all attention layers with low-rank tensorized UTA attention carrying mean/variance "
    "uncertainty streams, fine-tunes only UTA parameters on WikiText-2, and evaluates accuracy plus "
    "uncertainty trends across MuSiQue hop depths with saved diagnostic plots."
)

subprocess.run(["git", "checkout", "-B", BRANCH_NAME], check=True)
subprocess.run(["git", "add", "UTA_Attention_v1.ipynb"], check=True)
subprocess.run(["git", "commit", "-m", PR_TITLE], check=False)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=False)
subprocess.run(["git", "push", "-u", "origin", BRANCH_NAME], check=False)
subprocess.run(["gh", "pr", "create", "--title", PR_TITLE, "--body", PR_BODY], check=False)
print("Done. If commit/push/PR reported errors, authenticate git/gh and rerun this cell.")
